# GP Occupancy Mapping Sandbox

This notebook follows the style of the existing GP visibility sandbox and
the camera-projection investigation notebook:
- a compact self-contained methods notebook,
- explicit formulas and modeling assumptions,
- interpretable plots for the latent field, predictive uncertainty, and planner behavior,
- a clean comparison between map quality and EFE-relevant behavior.

The goal here is not to replace the direct `q(x)` thesis story. It is to
build a **structure-aware comparison model**: infer occupancy from
synthetic scan evidence with a GP, then derive a visibility/reliability
field from the GP occupancy mean.


## Model, Formulas, And Why This Notebook Is Only An Approximation

We model a latent occupancy field over planar position:

```math
f(\mathbf{x}) \sim \mathcal{GP}(0, k(\mathbf{x}, \mathbf{x}')),
\qquad
\mathbf{x} = [x, y]^	op.
```

Training evidence comes from synthetic scan rays:
- free-space samples along a ray get label `y_i = 0`,
- the first hit cell on a ray gets label `y_i = 1`.

The exact probabilistic occupancy-GP story would normally use a Bernoulli
likelihood with a sigmoid or probit link:

```math
p(y_i = 1 \mid f(\mathbf{x}_i)) = \sigma(f(\mathbf{x}_i)).
```

This notebook intentionally uses a simpler **GP regression surrogate** on
the binary labels, because it is lightweight, transparent, and consistent
with the existing `gp-visibility-efe-sandbox.ipynb` pattern:

```math
\mu_\*(X_\*) = K_{\*X}(K_{XX} + \sigma_n^2 I)^{-1} y,
```

```math
\Sigma_\*(X_\*) = K_{\*\*} - K_{\*X}(K_{XX} + \sigma_n^2 I)^{-1}K_{X\*}.
```

We then interpret the clipped posterior mean as an occupancy probability proxy:

```math
\hat p_{\mathrm{occ}}(\mathbf{x}) = \mathrm{clip}(\mu_\*(\mathbf{x}), 
arepsilon, 1-
arepsilon).
```

This is not exact GP classification. It is a practical surrogate.

Visibility is then derived by integrating occupancy along the camera-to-state ray:

```math
q_{\mathrm{GP-occ}}(x) =
\exp\left(-	au \; \overline{p_{\mathrm{occ}}}_{\mathrm{ray}}(x)
ight),
```

and the observation model remains the same as the other notebooks:

```math
R_{\mathrm{eff}}(x) = q(x)R_{\mathrm{good}} + (1-q(x))R_{\mathrm{bad}}.
```

Advantages of this route:
- explicitly geometry-aware,
- produces a predictive uncertainty map,
- easy to compare with hard occupancy and direct `q(x)` learning.

Disadvantages:
- one more layer of modeling assumptions than direct `q(x)`,
- sparse scan coverage can dominate the result,
- GP regression on binary labels is an approximation.


In [ ]:
import sys
from pathlib import Path
import math

repo_root = Path.cwd()
if not (repo_root / "scripts").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.interpolate import RegularGridInterpolator

from scripts.state_dependent_observation_helpers import (
    covariance_logdet_series,
    covariance_trace_series,
    evaluate_visibility_on_grid,
    make_action_library,
    make_default_camera,
    make_observation_fn,
    make_occupancy_grid,
    observation_covariance,
    process_covariance_from_rho,
    raycast_visibility_q,
    rollout_controls,
    score_action_library,
    simulate_receding_horizon,
)

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.2

rng = np.random.default_rng(12)


In [ ]:
# Scenario, hidden map, and synthetic scan evidence

camera = make_default_camera()
camera_xy = np.asarray(camera.cam_pos[:2], dtype=float)

true_grid = make_occupancy_grid(
    xmin=-4.5,
    xmax=2.8,
    ymin=-4.8,
    ymax=1.0,
    resolution=0.1,
    rectangles=[(-0.45, 0.45, -2.8, -1.0)],
    circles=None,
    border_occupancy=True,
)

goal_state = np.array([1.5, -1.2, 0.0], dtype=float)
representative_state = np.array([-1.0, -1.7, 0.0], dtype=float)


def sample_occ(grid, x, y):
    xs = grid.xs
    ys = grid.ys
    occ = grid.occupancy
    if x <= xs[0] or x >= xs[-1] or y <= ys[0] or y >= ys[-1]:
        return 1.0

    ix = int(np.searchsorted(xs, x, side="right") - 1)
    iy = int(np.searchsorted(ys, y, side="right") - 1)
    ix = max(0, min(ix, xs.shape[0] - 2))
    iy = max(0, min(iy, ys.shape[0] - 2))

    x0, x1 = xs[ix], xs[ix + 1]
    y0, y1 = ys[iy], ys[iy + 1]
    tx = 0.0 if x1 == x0 else (x - x0) / (x1 - x0)
    ty = 0.0 if y1 == y0 else (y - y0) / (y1 - y0)

    z00 = occ[iy, ix]
    z10 = occ[iy, ix + 1]
    z01 = occ[iy + 1, ix]
    z11 = occ[iy + 1, ix + 1]
    return float(
        (1.0 - ty) * ((1.0 - tx) * z00 + tx * z10)
        + ty * ((1.0 - tx) * z01 + tx * z11)
    )


def first_hit_point(grid, start, end, threshold=0.55, n=110):
    ts = np.linspace(0.0, 1.0, int(max(n, 2)))
    points = np.outer(1.0 - ts, start) + np.outer(ts, end)
    occ_values = np.asarray([sample_occ(grid, p[0], p[1]) for p in points], dtype=float)
    hit_idx = np.flatnonzero(occ_values >= float(threshold))
    if hit_idx.size == 0:
        return None, points
    idx = int(hit_idx[0])
    return points[idx].copy(), points[: idx + 1].copy()


def build_gp_dataset(grid, camera_xy, rng, *, n_rays=240, free_stride=6, max_free=360, max_occ=180):
    free_points = []
    occ_points = []
    example_rays = []

    for ray_idx in range(int(n_rays)):
        # Focus a subset of rays on the decision corridor so the GP sees
        # evidence relevant to the visibility tradeoff near the occluder.
        if ray_idx % 5 == 0:
            end = np.array([
                rng.uniform(-1.2, 1.8),
                rng.uniform(-2.5, -0.6),
            ], dtype=float)
        else:
            end = np.array([
                rng.uniform(grid.xs.min() + 0.1, grid.xs.max() - 0.1),
                rng.uniform(grid.ys.min() + 0.1, grid.ys.max() - 0.1),
            ], dtype=float)

        hit_point, traversed_points = first_hit_point(grid, camera_xy, end, threshold=0.55, n=110)
        if hit_point is not None:
            occ_points.append(hit_point.copy())

        free_segment = traversed_points[:-1] if hit_point is not None else traversed_points
        free_points.extend(free_segment[::free_stride])

        if ray_idx < 24:
            example_rays.append({
                "points": traversed_points.copy(),
                "hit": None if hit_point is None else hit_point.copy(),
            })

    free_points = np.asarray(free_points, dtype=float)
    occ_points = np.asarray(occ_points, dtype=float) if occ_points else np.empty((0, 2), dtype=float)

    if free_points.shape[0] > int(max_free):
        idx = rng.choice(free_points.shape[0], size=int(max_free), replace=False)
        free_points = free_points[idx]
    if occ_points.shape[0] > int(max_occ):
        idx = rng.choice(occ_points.shape[0], size=int(max_occ), replace=False)
        occ_points = occ_points[idx]

    X_train = np.vstack([free_points, occ_points])
    y_train = np.concatenate([
        np.zeros(free_points.shape[0], dtype=float),
        np.ones(occ_points.shape[0], dtype=float),
    ])
    return X_train, y_train, free_points, occ_points, example_rays


X_train, y_train, free_points, occ_points, example_rays = build_gp_dataset(true_grid, camera_xy, rng)

print("Camera xy:", camera_xy)
print("Training points:", X_train.shape[0])
print("Free / occupied:", free_points.shape[0], "/", occ_points.shape[0])
print("Occupied label rate:", float(y_train.mean()))
print("Representative start true visibility:", float(raycast_visibility_q(representative_state, true_grid, camera_xy, tau=8.0, n_samples=100)))


In [ ]:
# GP occupancy model

class SimpleRBFGP:
    def __init__(self, length_scale=0.42, signal_var=1.0, noise_var=0.06, jitter=1e-8):
        self.length_scale = float(length_scale)
        self.signal_var = float(signal_var)
        self.noise_var = float(noise_var)
        self.jitter = float(jitter)
        self.X_train = None
        self.y_mean = None
        self.L = None
        self.alpha = None

    def _kernel(self, Xa, Xb):
        Xa = np.asarray(Xa, dtype=float)
        Xb = np.asarray(Xb, dtype=float)
        d2 = np.sum((Xa[:, None, :] - Xb[None, :, :]) ** 2, axis=2)
        ls2 = max(self.length_scale ** 2, 1e-12)
        return self.signal_var * np.exp(-0.5 * d2 / ls2)

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1)
        self.X_train = X
        self.y_mean = float(y.mean())
        y0 = y - self.y_mean

        K = self._kernel(X, X)
        K = K + (self.noise_var + self.jitter) * np.eye(X.shape[0])
        self.L = np.linalg.cholesky(K)
        self.alpha = np.linalg.solve(self.L.T, np.linalg.solve(self.L, y0))
        return self

    def predict(self, X, return_std=False):
        X = np.asarray(X, dtype=float)
        Ks = self._kernel(X, self.X_train)
        mu = self.y_mean + Ks @ self.alpha

        if not return_std:
            return mu

        v = np.linalg.solve(self.L, Ks.T)
        var = np.maximum(self.signal_var - np.sum(v * v, axis=0), 1e-12)
        return mu, np.sqrt(var)


gp_occ = SimpleRBFGP(length_scale=0.42, signal_var=1.0, noise_var=0.06).fit(X_train, y_train)

xs = np.linspace(true_grid.xs.min(), true_grid.xs.max(), 70)
ys = np.linspace(true_grid.ys.min(), true_grid.ys.max(), 56)
Xg, Yg = np.meshgrid(xs, ys)
XY = np.column_stack([Xg.ravel(), Yg.ravel()])

occ_mu, occ_std = gp_occ.predict(XY, return_std=True)
P_occ_gp = np.clip(occ_mu.reshape(Xg.shape), 1e-4, 1.0 - 1e-4)
P_occ_std = occ_std.reshape(Xg.shape)

true_occ_interp = RegularGridInterpolator(
    (true_grid.ys, true_grid.xs),
    true_grid.occupancy,
    bounds_error=False,
    fill_value=1.0,
)
P_occ_true = true_occ_interp(np.column_stack([Yg.ravel(), Xg.ravel()])).reshape(Xg.shape)

gp_occ_interp = RegularGridInterpolator(
    (ys, xs),
    P_occ_gp,
    bounds_error=False,
    fill_value=0.5,
)

def p_occ_gp_xy(xy):
    val = float(gp_occ_interp([[float(xy[1]), float(xy[0])]])[0])
    return float(np.clip(val, 1e-4, 1.0 - 1e-4))


def q_gp_occ(state, tau=8.0, n_samples=90):
    start = camera_xy
    end = np.asarray(state[:2], dtype=float)
    ts = np.linspace(0.0, 1.0, int(max(n_samples, 2)))
    points = np.outer(1.0 - ts, start) + np.outer(ts, end)
    occ_vals = np.clip(gp_occ_interp(np.column_stack([points[:, 1], points[:, 0]])), 1e-4, 1.0)
    return float(np.clip(np.exp(-float(tau) * float(np.mean(occ_vals))), 0.0, 1.0))


def q_true_occ(state, tau=8.0, n_samples=90):
    return raycast_visibility_q(state, true_grid, camera_xy, tau=tau, n_samples=n_samples)


occ_rmse = float(np.sqrt(np.mean((P_occ_gp - P_occ_true) ** 2)))
print("Occupancy RMSE against hidden map:", occ_rmse)
print("Predictive std mean:", float(P_occ_std.mean()))
print("q_true(representative):", q_true_occ(representative_state))
print("q_gp(representative):  ", q_gp_occ(representative_state))


In [ ]:
# Visualize scan evidence, GP occupancy mean, and predictive std

fig, axes = plt.subplots(1, 3, figsize=(16, 5.0), sharex=True, sharey=True)

axes[0].imshow(true_grid.occupancy, extent=true_grid.extent, origin="lower", cmap="Greys", aspect="equal")
for record in example_rays:
    points = record["points"]
    axes[0].plot(points[:, 0], points[:, 1], color="tab:orange", alpha=0.18, linewidth=1.2)
    if record["hit"] is not None:
        axes[0].scatter(record["hit"][0], record["hit"][1], s=16, color="tab:red", alpha=0.45)
axes[0].scatter(free_points[:, 0], free_points[:, 1], s=8, color="tab:green", alpha=0.25, label="free samples")
axes[0].scatter(occ_points[:, 0], occ_points[:, 1], s=14, color="tab:red", alpha=0.6, label="hit samples")
axes[0].scatter(camera_xy[0], camera_xy[1], marker="^", s=80, color="tab:blue", label="camera")
axes[0].set_title("Hidden map and GP training evidence")
axes[0].set_xlabel("x [m]")
axes[0].set_ylabel("y [m]")
axes[0].legend(loc="lower left", fontsize=8)

im1 = axes[1].imshow(P_occ_gp, extent=true_grid.extent, origin="lower", cmap="viridis", aspect="equal", vmin=0.0, vmax=1.0)
axes[1].scatter(camera_xy[0], camera_xy[1], marker="^", s=80, color="white", edgecolor="black")
axes[1].set_title("GP occupancy mean")
axes[1].set_xlabel("x [m]")
plt.colorbar(im1, ax=axes[1], label="occupancy probability")

im2 = axes[2].imshow(P_occ_std, extent=true_grid.extent, origin="lower", cmap="magma", aspect="equal")
axes[2].scatter(camera_xy[0], camera_xy[1], marker="^", s=80, color="white", edgecolor="black")
axes[2].set_title("GP predictive std")
axes[2].set_xlabel("x [m]")
plt.colorbar(im2, ax=axes[2], label="std")

for ax in axes:
    ax.set_xlim(-4.5, 2.5)
    ax.set_ylim(-4.8, 0.8)

plt.tight_layout()
plt.show()


## Interpreting The Kernel Length Scale

In a GP occupancy model, the kernel length scale controls how strongly
evidence at one location influences nearby cells:
- small length scale: sharper local detail, but more variance and less interpolation,
- large length scale: smoother occupancy field, but risk of washing out narrow structures.

The next plot compares a short and long length scale on a 1D slice. This
is the occupancy analogue of the sensitivity investigations in the camera
projection notebook: the point is not just to produce one map, but to show
how the modeling assumption shapes the result.


In [ ]:
gp_short = SimpleRBFGP(length_scale=0.25, signal_var=1.0, noise_var=0.06).fit(X_train, y_train)
gp_long = SimpleRBFGP(length_scale=0.85, signal_var=1.0, noise_var=0.06).fit(X_train, y_train)

x_slice = np.linspace(true_grid.xs.min(), true_grid.xs.max(), 220)
slice_states_xy = np.column_stack([x_slice, np.full_like(x_slice, representative_state[1])])
slice_true = np.asarray([sample_occ(true_grid, x, representative_state[1]) for x in x_slice], dtype=float)
slice_main = np.clip(gp_occ.predict(slice_states_xy), 1e-4, 1.0 - 1e-4)
slice_short = np.clip(gp_short.predict(slice_states_xy), 1e-4, 1.0 - 1e-4)
slice_long = np.clip(gp_long.predict(slice_states_xy), 1e-4, 1.0 - 1e-4)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.4), constrained_layout=True)

axes[0].plot(x_slice, slice_true, linewidth=2.5, color="black", label="true occupancy")
axes[0].plot(x_slice, slice_short, linewidth=2, label="GP short ls = 0.25")
axes[0].plot(x_slice, slice_main, linewidth=2, label="GP main ls = 0.42")
axes[0].plot(x_slice, slice_long, linewidth=2, label="GP long ls = 0.85")
axes[0].axvline(representative_state[0], color="black", linestyle=":", linewidth=1)
axes[0].set_title(f"Occupancy slice at y = {representative_state[1]:.1f} m")
axes[0].set_xlabel("x [m]")
axes[0].set_ylabel("occupancy probability")
axes[0].legend(loc="upper right", fontsize=8)

axes[1].imshow(P_occ_gp, extent=true_grid.extent, origin="lower", cmap="viridis", aspect="equal", vmin=0.0, vmax=1.0, alpha=0.92)
axes[1].plot(x_slice, np.full_like(x_slice, representative_state[1]), color="white", linewidth=2, linestyle="--", label="slice")
axes[1].scatter(representative_state[0], representative_state[1], s=60, color="white", edgecolor="black", label="representative state")
axes[1].scatter(camera_xy[0], camera_xy[1], marker="^", s=80, color="white", edgecolor="black", label="camera")
axes[1].set_title("Where the 1D slice is taken")
axes[1].set_xlabel("x [m]")
axes[1].set_ylabel("y [m]")
axes[1].legend(loc="upper right", fontsize=8)

plt.show()


## From GP Occupancy To A Planner-Facing Visibility Field

The GP mean is not used directly by the planner. It is first converted to
a ray-based visibility field `q(x)`. This mirrors the structure-aware path
from the occupancy notebooks, but now the occupancy map itself is learned
by a GP.

A useful interpretation:
- the occupancy mean says which parts of the world are likely blocked,
- the predictive standard deviation says where the map is still uncertain,
- the ray integral turns that geometric belief into a sensing-reliability proxy.

This still differs from direct learned `q(x)`:
- direct `q(x)` learns the sensing outcome itself,
- GP occupancy learns environment structure first and only then derives visibility.


In [ ]:
xs_vis = np.linspace(true_grid.xs.min(), true_grid.xs.max(), 70)
ys_vis = np.linspace(true_grid.ys.min(), true_grid.ys.max(), 56)
Xv, Yv = np.meshgrid(xs_vis, ys_vis)
states_vis = np.stack([Xv, Yv, np.zeros_like(Xv)], axis=-1)

q_true_map = evaluate_visibility_on_grid(states_vis, q_true_occ)
q_gp_map = evaluate_visibility_on_grid(states_vis, q_gp_occ)
q_abs_err = np.abs(q_gp_map - q_true_map)

q_slice_true = np.asarray([q_true_occ(np.array([x, representative_state[1], 0.0])) for x in x_slice], dtype=float)
q_slice_gp = np.asarray([q_gp_occ(np.array([x, representative_state[1], 0.0])) for x in x_slice], dtype=float)

fig, axes = plt.subplots(2, 2, figsize=(13, 10), constrained_layout=True)

im0 = axes[0, 0].imshow(q_true_map, extent=true_grid.extent, origin="lower", cmap="viridis", aspect="equal", vmin=0.0, vmax=1.0)
axes[0, 0].scatter(camera_xy[0], camera_xy[1], marker="^", s=80, color="white", edgecolor="black")
axes[0, 0].set_title("True-map soft visibility")
plt.colorbar(im0, ax=axes[0, 0], label="q(x)")

im1 = axes[0, 1].imshow(q_gp_map, extent=true_grid.extent, origin="lower", cmap="viridis", aspect="equal", vmin=0.0, vmax=1.0)
axes[0, 1].scatter(camera_xy[0], camera_xy[1], marker="^", s=80, color="white", edgecolor="black")
axes[0, 1].set_title("GP-occupancy-derived visibility")
plt.colorbar(im1, ax=axes[0, 1], label="q(x)")

im2 = axes[1, 0].imshow(q_abs_err, extent=true_grid.extent, origin="lower", cmap="magma", aspect="equal")
axes[1, 0].scatter(camera_xy[0], camera_xy[1], marker="^", s=80, color="white", edgecolor="black")
axes[1, 0].set_title("Absolute visibility error")
plt.colorbar(im2, ax=axes[1, 0], label="|q_gp - q_true|")

axes[1, 1].plot(x_slice, q_slice_true, linewidth=2.5, color="black", label="true map")
axes[1, 1].plot(x_slice, q_slice_gp, linewidth=2.0, linestyle="--", label="GP occupancy")
axes[1, 1].axvline(representative_state[0], color="black", linestyle=":", linewidth=1)
axes[1, 1].set_title(f"Visibility slice at y = {representative_state[1]:.1f} m")
axes[1, 1].set_xlabel("x [m]")
axes[1, 1].set_ylabel("q(x)")
axes[1, 1].legend()

for ax in axes.ravel()[:3]:
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")

plt.show()


In [ ]:
# Planner comparison: constant R vs true map vs GP occupancy

g = make_observation_fn(camera, obs_mode="uv")
R_good = observation_covariance("uv", uv_std=2.5)
R_bad = observation_covariance("uv", uv_std=20.0)
dt = 0.2
Q = process_covariance_from_rho(dt=dt, rho_xy=1e-2)
cov0 = np.diag([0.18 ** 2, 0.18 ** 2, 0.10 ** 2])

actions = make_action_library(
    v_values=(0.0, 0.18, 0.28, 0.38),
    w_values=(-1.2, -0.8, -0.35, 0.0, 0.35, 0.8, 1.2),
)

common_kwargs = dict(
    dt=dt,
    Q=Q,
    g=g,
    goal_obs=g(goal_state),
    goal_obs_cov=np.diag([55.0 ** 2, 55.0 ** 2]),
    R_good=R_good,
    R_bad=R_bad,
    horizon=8,
    risk_weight=1.0,
    ambiguity_weight=1.4,
    control_weight=0.05,
    approx="ET2",
    add_ambiguity=True,
)

benchmark_states = np.array([
    [-1.2, -1.8, 0.0],
    [-1.0, -1.7, 0.0],
    [-0.9, -1.6, 0.0],
    [-0.8, -1.5, 0.0],
    [-1.1, -1.3, 0.0],
    [-0.9, -1.2, 0.0],
], dtype=float)


def turn_class(control):
    omega = float(control[1])
    if abs(omega) < 0.2:
        return "straight"
    return "left" if omega > 0.0 else "right"


rows = []
for state in benchmark_states:
    constant_best = score_action_library(state, cov0, actions, q_fn=None, **common_kwargs)[0]
    true_best = score_action_library(state, cov0, actions, q_fn=q_true_occ, **common_kwargs)[0]
    gp_best = score_action_library(state, cov0, actions, q_fn=q_gp_occ, **common_kwargs)[0]
    rows.append({
        "x": state[0],
        "y": state[1],
        "q_true": float(q_true_occ(state)),
        "q_gp": float(q_gp_occ(state)),
        "constant_action": tuple(np.round(constant_best.control, 2)),
        "true_action": tuple(np.round(true_best.control, 2)),
        "gp_action": tuple(np.round(gp_best.control, 2)),
        "constant_turn": turn_class(constant_best.control),
        "true_turn": turn_class(true_best.control),
        "gp_turn": turn_class(gp_best.control),
        "gp_same_turn_as_true": turn_class(gp_best.control) == turn_class(true_best.control),
        "constant_same_turn_as_true": turn_class(constant_best.control) == turn_class(true_best.control),
    })

benchmark_df = pd.DataFrame(rows)
print("GP same-turn agreement:", float(benchmark_df["gp_same_turn_as_true"].mean()))
print("Constant-R same-turn agreement:", float(benchmark_df["constant_same_turn_as_true"].mean()))
benchmark_df


In [ ]:
mean0 = representative_state.copy()

preview_candidates = {
    "constant R": score_action_library(mean0, cov0, actions, q_fn=None, **common_kwargs)[0],
    "true-map q(x)": score_action_library(mean0, cov0, actions, q_fn=q_true_occ, **common_kwargs)[0],
    "GP-occ q(x)": score_action_library(mean0, cov0, actions, q_fn=q_gp_occ, **common_kwargs)[0],
}

preview_rollouts = {}
for label, best in preview_candidates.items():
    preview_controls = np.repeat(best.control[None, :], common_kwargs["horizon"], axis=0)
    preview_states = rollout_controls(mean0, preview_controls, dt)
    preview_rollouts[label] = {
        "control": best.control.copy(),
        "states": preview_states,
        "q_true_along_path": np.asarray([q_true_occ(state) for state in preview_states[1:]], dtype=float),
        "q_gp_along_path": np.asarray([q_gp_occ(state) for state in preview_states[1:]], dtype=float),
    }

runs = {
    "constant R": simulate_receding_horizon(
        mean0,
        cov0,
        actions,
        q_fn=None,
        n_steps=10,
        **common_kwargs,
    ),
    "true-map q(x)": simulate_receding_horizon(
        mean0,
        cov0,
        actions,
        q_fn=q_true_occ,
        n_steps=10,
        **common_kwargs,
    ),
    "GP-occ q(x)": simulate_receding_horizon(
        mean0,
        cov0,
        actions,
        q_fn=q_gp_occ,
        n_steps=10,
        **common_kwargs,
    ),
}

summary_df = pd.DataFrame([
    {
        "model": label,
        "first_action": tuple(np.round(run["controls"][0], 2)),
        "planned_first_action": tuple(np.round(preview_candidates[label].control, 2)),
        "planned_min_q_true": float(preview_rollouts[label]["q_true_along_path"].min()),
        "mean_q_runtime": float(run["q_values"].mean()) if len(run["q_values"]) else 1.0,
        "min_q_runtime": float(run["q_values"].min()) if len(run["q_values"]) else 1.0,
        "final_trace": float(covariance_trace_series(run["covs"])[-1]),
        "final_logdet": float(covariance_logdet_series(run["covs"])[-1]),
    }
    for label, run in runs.items()
])

print("Representative rollout summary:")
summary_df


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10), constrained_layout=True)

axes[0, 0].imshow(q_gp_map, extent=true_grid.extent, origin="lower", cmap="viridis", aspect="equal", alpha=0.85, vmin=0.0, vmax=1.0)
axes[0, 0].imshow(true_grid.occupancy, extent=true_grid.extent, origin="lower", cmap="Greys", aspect="equal", alpha=0.18)
for label, run in runs.items():
    axes[0, 0].plot(run["means"][:, 0], run["means"][:, 1], linewidth=2.0, label=label)
axes[0, 0].scatter(mean0[0], mean0[1], s=60, color="white", edgecolor="black", label="start")
axes[0, 0].scatter(goal_state[0], goal_state[1], marker="*", s=180, color="tab:red", label="goal")
axes[0, 0].scatter(camera_xy[0], camera_xy[1], marker="^", s=80, color="white", edgecolor="black", label="camera")
axes[0, 0].set_title("Representative rollout from a visible start")
axes[0, 0].set_xlabel("x [m]")
axes[0, 0].set_ylabel("y [m]")
axes[0, 0].legend(loc="lower right")

for label, preview in preview_rollouts.items():
    axes[0, 1].plot(preview["q_true_along_path"], marker="o", linewidth=2, label=label)
axes[0, 1].set_title("Visibility along first planned horizon (true map)")
axes[0, 1].set_xlabel("step")
axes[0, 1].set_ylabel("q(x)")
axes[0, 1].legend()

for label, run in runs.items():
    axes[1, 0].plot(run["ambiguity_terms"], marker="o", linewidth=2, label=label)
axes[1, 0].set_title("Ambiguity contribution")
axes[1, 0].set_xlabel("step")
axes[1, 0].set_ylabel("ambiguity")
axes[1, 0].legend()

for label, run in runs.items():
    axes[1, 1].plot(covariance_trace_series(run["covs"]), marker="o", linewidth=2, label=label)
axes[1, 1].set_title("Belief covariance trace")
axes[1, 1].set_xlabel("step")
axes[1, 1].set_ylabel("trace(P)")
axes[1, 1].legend()

plt.show()


## Takeaways, Alternatives, And Sources

Main takeaways from this sandbox:
- GP occupancy mapping can recover enough geometric structure to influence the planner in a useful way,
- the predictive uncertainty map reveals where sparse scan coverage is dominating the result,
- planner agreement can improve even when occupancy RMSE is still far from perfect,
- this remains a comparison route, not the cleanest primary thesis mechanism.

Why it is not the primary thesis mechanism:
- the direct question is whether state-dependent observation quality changes EFE behavior,
- GP occupancy adds an intermediate latent map model and a ray-integration step,
- that extra structure is useful for comparison, but not necessary to answer the main claim.

Better approximations if you want to push this further:
- GP classification with Laplace or expectation propagation,
- sparse GP occupancy mapping for larger datasets,
- uncertainty-aware planning that uses both occupancy mean and occupancy variance,
- multi-view scan generation instead of one fixed external camera.

Sources:
- C. E. Rasmussen and C. K. I. Williams, *Gaussian Processes for Machine Learning*, 2006.
- A. Elfes, *Occupancy Grids: A Stochastic Spatial Representation for Active Robot Perception*, 1989.
- S. Thrun, W. Burgard, D. Fox, *Probabilistic Robotics*, 2005.
- Local style references: `scripts/gp-visibility-efe-sandbox.ipynb` and `scripts/camera-projection-investigation.ipynb`.
